# Lecture 6 Hands-on: Tokenization



- Character tokenizer
- Word tokenizer
- Subword tokenizer intuition
- BPE from scratch
- WordPiece using Hugging Face
- SentencePiece
- Compare token counts
- See the impact on LLM/RAG

## 0. Install dependencies

In [ ]:
!pip install transformers sentencepiece tokenizers

## Part 1 — Character Tokenization

### Cell 1

In [54]:
text = "I love machine learning"

characters = list(text)

print(characters)
print("Number of characters:", len(characters))

['I', ' ', 'l', 'o', 'v', 'e', ' ', 'm', 'a', 'c', 'h', 'i', 'n', 'e', ' ', 'l', 'e', 'a', 'r', 'n', 'i', 'n', 'g']
Number of characters: 23


## Part 2 — Build a Character Vocabulary

### Cell 2

In [55]:
text = """
machine learning is powerful
machine learning is useful
deep learning is powerful
"""

chars = sorted(set(text))

print("Vocabulary:")
print(chars)

print("Vocabulary size:", len(chars))

Vocabulary:
['\n', ' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 'u', 'w']
Vocabulary size: 19


Now create mappings.

### Cell 3

In [56]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

print(stoi)

{'\n': 0, ' ': 1, 'a': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'l': 10, 'm': 11, 'n': 12, 'o': 13, 'p': 14, 'r': 15, 's': 16, 'u': 17, 'w': 18}


Convert text → IDs.

### Cell 4

In [57]:
encoded = [stoi[ch] for ch in "machine"]

print(encoded)

[11, 2, 3, 8, 9, 12, 5]


Decode it:

In [58]:
decoded = "".join(itos[i] for i in encoded)

print(decoded)

machine


Implemented the basic idea behind:

```
Text
 ↓
Tokens
 ↓
Token IDs
```

## Part 3 — Word Tokenization

Now move from characters to words.

### Cell 5

In [59]:
text = """
Machine learning is powerful.
Machine learning is useful.
Deep learning is powerful.
"""

words = text.lower().split()

print(words)
print("Number of tokens:", len(words))

['machine', 'learning', 'is', 'powerful.', 'machine', 'learning', 'is', 'useful.', 'deep', 'learning', 'is', 'powerful.']
Number of tokens: 12


Create vocabulary:

In [60]:
vocab = sorted(set(words))

print(vocab)
print("Vocabulary size:", len(vocab))

['deep', 'is', 'learning', 'machine', 'powerful.', 'useful.']
Vocabulary size: 6


Create token IDs:

In [61]:
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}

print(word_to_id)

{'deep': 0, 'is': 1, 'learning': 2, 'machine': 3, 'powerful.': 4, 'useful.': 5}


Encode:

In [62]:
sentence = "machine learning is powerful."

ids = [word_to_id[word] for word in sentence.split()]

print(ids)

[3, 2, 1, 4]


Decode:

In [63]:
decoded = " ".join(id_to_word[i] for i in ids)

print(decoded)

machine learning is powerful.


## Part 4 — Demonstrate the OOV Problem



### Cell 6

In [64]:
sentence = "machine learning is amazing"

for word in sentence.split():
    if word in word_to_id:
        print(word, "→", word_to_id[word])
    else:
        print(word, "→ <UNK>")

machine → 3
learning → 2
is → 1
amazing → <UNK>


## Part 5 — Simple Subword Intuition

Before implementing BPE, give students a small example.

In [65]:
words = [
    "play",
    "played",
    "playing",
    "player",
    "playful"
]

for word in words:
    print(word)

play
played
playing
player
playful


## Part 6 — BPE From Scratch

This is the most important hands-on portion.

We will implement a simplified educational BPE algorithm.

### Step 1 — Training corpus

### Cell 7

In [66]:
corpus = [
    "low",
    "lower",
    "lowest",
    "low",
    "lower",
    "newest",
    "newer"
]

print(corpus)

['low', 'lower', 'lowest', 'low', 'lower', 'newest', 'newer']


Represent each word as characters.

In [67]:
word_tokens = {
    word: list(word) + ["</w>"]
    for word in corpus
}

print(word_tokens)

{'low': ['l', 'o', 'w', '</w>'], 'lower': ['l', 'o', 'w', 'e', 'r', '</w>'], 'lowest': ['l', 'o', 'w', 'e', 's', 't', '</w>'], 'newest': ['n', 'e', 'w', 'e', 's', 't', '</w>'], 'newer': ['n', 'e', 'w', 'e', 'r', '</w>']}


You may explain:

```
low

l o w </w>
```

`</w>` indicates the end of a word in this simplified implementation.

## Part 7 — Count Adjacent Pairs

### Cell 8

In [68]:
from collections import Counter

def get_pair_counts(word_tokens):
    pair_counts = Counter()

    for tokens in word_tokens.values():
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pair_counts[pair] += 1

    return pair_counts

Run:

In [69]:
pair_counts = get_pair_counts(word_tokens)

for pair, count in pair_counts.most_common():
    print(pair, count)

('w', 'e') 4
('l', 'o') 3
('o', 'w') 3
('e', 'r') 2
('r', '</w>') 2
('e', 's') 2
('s', 't') 2
('t', '</w>') 2
('n', 'e') 2
('e', 'w') 2
('w', '</w>') 1


## Part 8 — Merge the Most Frequent Pair

### Cell 9

In [70]:
def merge_pair(word_tokens, pair_to_merge):
    
    new_word_tokens = {}

    for word, tokens in word_tokens.items():
        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == pair_to_merge[0]
                and tokens[i + 1] == pair_to_merge[1]
            ):
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        new_word_tokens[word] = new_tokens

    return new_word_tokens

Now perform one merge:

In [71]:
pair_counts = get_pair_counts(word_tokens)

best_pair = pair_counts.most_common(1)[0][0]

print("Most frequent pair:", best_pair)

word_tokens = merge_pair(
    word_tokens,
    best_pair
)

print(word_tokens)

Most frequent pair: ('w', 'e')
{'low': ['l', 'o', 'w', '</w>'], 'lower': ['l', 'o', 'we', 'r', '</w>'], 'lowest': ['l', 'o', 'we', 's', 't', '</w>'], 'newest': ['n', 'e', 'we', 's', 't', '</w>'], 'newer': ['n', 'e', 'we', 'r', '</w>']}


## Part 9 — Train BPE

Now put the process inside a loop.

### Cell 10

In [72]:
def train_bpe(corpus, num_merges=10):

    word_tokens = {
        word: list(word) + ["</w>"]
        for word in corpus
    }

    merges = []

    for step in range(num_merges):

        pair_counts = get_pair_counts(word_tokens)

        if not pair_counts:
            break

        best_pair, count = pair_counts.most_common(1)[0]

        merges.append(best_pair)

        word_tokens = merge_pair(
            word_tokens,
            best_pair
        )

        print(
            f"Merge {step + 1}: "
            f"{best_pair} → frequency {count}"
        )

    return word_tokens, merges

Run:

In [73]:
final_tokens, merges = train_bpe(
    corpus,
    num_merges=10
)

print("\nLearned merges:")
for merge in merges:
    print(merge)

Merge 1: ('w', 'e') → frequency 4
Merge 2: ('l', 'o') → frequency 3
Merge 3: ('lo', 'we') → frequency 2
Merge 4: ('r', '</w>') → frequency 2
Merge 5: ('s', 't') → frequency 2
Merge 6: ('st', '</w>') → frequency 2
Merge 7: ('n', 'e') → frequency 2
Merge 8: ('ne', 'we') → frequency 2
Merge 9: ('lo', 'w') → frequency 1
Merge 10: ('low', '</w>') → frequency 1

Learned merges:
('w', 'e')
('l', 'o')
('lo', 'we')
('r', '</w>')
('s', 't')
('st', '</w>')
('n', 'e')
('ne', 'we')
('lo', 'w')
('low', '</w>')


## Part 10 — Observe BPE Learning

Print the final vocabulary representation:

In [74]:
print("\nFinal tokenization:")

for word, tokens in final_tokens.items():
    print(word, "→", tokens)


Final tokenization:
low → ['low</w>']
lower → ['lowe', 'r</w>']
lowest → ['lowe', 'st</w>']
newest → ['newe', 'st</w>']
newer → ['newe', 'r</w>']


## Part 11 — BPE Tokenization of a New Word

Now demonstrate an unseen word.

For example:

In [75]:
new_word = "lowering"

print(list(new_word))

['l', 'o', 'w', 'e', 'r', 'i', 'n', 'g']


## Part 12 — Use Real BPE Tokenizer

Now transition:

"We implemented a simplified version. Real-world tokenizers are more sophisticated. Let's use a production tokenizer."

Use a Hugging Face tokenizer.

### Cell 11

In [78]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "gpt2"
)

Tokenize:

In [81]:
text = "TTransformer is brain of large language models."

tokens = tokenizer.tokenize(text)

print(tokens)

['T', 'Trans', 'former', 'Ġis', 'Ġbrain', 'Ġof', 'Ġlarge', 'Ġlanguage', 'Ġmodels', '.']


Token IDs:

In [82]:
ids = tokenizer.encode(text)

print(ids)
#[30642, 1634, 318, 1593, 329, 1588, 3303, 4981, 13]

[51, 8291, 16354, 318, 3632, 286, 1588, 3303, 4981, 13]


Number of tokens:

In [83]:
print("Number of tokens:", len(ids))

Number of tokens: 10


Decode:

In [84]:
decoded = tokenizer.decode(ids)

print(decoded)

TTransformer is brain of large language models.


## Part 13 — See Tokenization Visually

### Cell 12

In [85]:
for token, token_id in zip(
    tokenizer.tokenize(text),
    tokenizer.encode(text)
):
    print(f"{token:20} → {token_id}")

T                    → 51
Trans                → 8291
former               → 16354
Ġis                  → 318
Ġbrain               → 3632
Ġof                  → 286
Ġlarge               → 1588
Ġlanguage            → 3303
Ġmodels              → 4981
.                    → 13


This is an excellent classroom demonstration.

**Ask:**

Why isn't every word necessarily one token?

Then try:

In [86]:
examples = [
    "playing",
    "played",
    "playfulness",
    "unbelievable",
    "machine learning",
    "ChatGPT",
    "😊"
]

for text in examples:

    tokens = tokenizer.tokenize(text)

    print(text)
    print(tokens)
    print()

playing
['playing']

played
['played']

playfulness
['play', 'fulness']

unbelievable
['un', 'bel', 'iev', 'able']

machine learning
['machine', 'Ġlearning']

ChatGPT
['Chat', 'G', 'PT']

😊
['ðŁĺ', 'Ĭ']



## Part 14 — WordPiece with BERT

Now introduce WordPiece practically.

### Cell 13

In [87]:
bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

Try:

In [88]:
words = [
    "playing",
    "played",
    "playful",
    "unbelievable",
    "tokenization",
    "transformers"
]

for word in words:

    tokens = bert_tokenizer.tokenize(word)

    print(f"{word:20} → {tokens}")

playing              → ['playing']
played               → ['played']
playful              → ['playful']
unbelievable         → ['unbelievable']
tokenization         → ['token', '##ization']
transformers         → ['transformers']


## Part 15 — Explain ##

In [34]:
text = "unbelievable"

tokens = bert_tokenizer.tokenize(text)

print(tokens)

['unbelievable']




`##piece`

means that the piece is intended to continue the current word.

For example:

`play + ##ing`

means:

`playing`

rather than two separate words.

## Part 16 — Compare BPE and WordPiece

### Cell 14

In [35]:
text = """
Machine learning is changing the world.
Tokenization is an important part of large language models.
"""

gpt_tokens = tokenizer.tokenize(text)
bert_tokens = bert_tokenizer.tokenize(text)

print("GPT-style tokenizer:")
print(gpt_tokens)

print("\nBERT WordPiece tokenizer:")
print(bert_tokens)

GPT-style tokenizer:
['Ċ', 'Machine', 'Ġlearning', 'Ġis', 'Ġchanging', 'Ġthe', 'Ġworld', '.', 'Ċ', 'Token', 'ization', 'Ġis', 'Ġan', 'Ġimportant', 'Ġpart', 'Ġof', 'Ġlarge', 'Ġlanguage', 'Ġmodels', '.', 'Ċ']

BERT WordPiece tokenizer:
['machine', 'learning', 'is', 'changing', 'the', 'world', '.', 'token', '##ization', 'is', 'an', 'important', 'part', 'of', 'large', 'language', 'models', '.']


Then:

In [36]:
print("GPT token count:", len(gpt_tokens))
print("BERT token count:", len(bert_tokens))

GPT token count: 21
BERT token count: 18




The two tokenizers can produce different token boundaries and token counts because:
- They use different vocabularies.
- They use different tokenization schemes.
- They were trained differently.

## Part 17 — SentencePiece

Install:
```
pip install sentencepiece
```

Then:

In [37]:
from transformers import AutoTokenizer

Use a SentencePiece-based model such as T5.

### Cell 15

In [38]:
sp_tokenizer = AutoTokenizer.from_pretrained(
    "t5-small"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Try:

In [39]:
text = "I love machine learning"

tokens = sp_tokenizer.tokenize(text)

print(tokens)

['▁I', '▁love', '▁machine', '▁learning']


Special word-boundary marker:

`▁`

For example, conceptually:
```
▁I
▁love
▁machine
▁learning
```

Explain:

SentencePiece does not require us to split the sentence into words using spaces first.

## Part 18 — SentencePiece With Other Languages

This is a nice demonstration for your students.

### Cell 16

In [40]:
texts = [
    "I love machine learning",
    "मुझे मशीन लर्निंग पसंद है",
    "我喜欢机器学习",
    "人工知能が好きです"
]

for text in texts:

    tokens = sp_tokenizer.tokenize(text)

    print("\nText:", text)
    print("Tokens:", tokens)
    print("Number of tokens:", len(tokens))


Text: I love machine learning
Tokens: ['▁I', '▁love', '▁machine', '▁learning']
Number of tokens: 4

Text: मुझे मशीन लर्निंग पसंद है
Tokens: ['▁', 'मुझे', '▁', 'मशीन', '▁', 'लर्निंग', '▁', 'पसंद', '▁', 'है']
Number of tokens: 10

Text: 我喜欢机器学习
Tokens: ['▁', '我喜欢机器学习']
Number of tokens: 2

Text: 人工知能が好きです
Tokens: ['▁', '人工知能が好きです']
Number of tokens: 2


Discuss:

Why can a tokenizer that works directly with raw text be useful for multilingual applications?

## Part 19 — Token Count Experiment

This is particularly relevant to GenAI.

### Cell 17

In [41]:
texts = [
    "I am learning artificial intelligence.",
    "Artificial intelligence is transforming healthcare.",
    "Tokenization is fundamental to large language models.",
    "The quick brown fox jumps over the lazy dog."
]

for text in texts:

    tokens = tokenizer.tokenize(text)

    print("\nText:", text)
    print("Tokens:", tokens)
    print("Token count:", len(tokens))


Text: I am learning artificial intelligence.
Tokens: ['I', 'Ġam', 'Ġlearning', 'Ġartificial', 'Ġintelligence', '.']
Token count: 6

Text: Artificial intelligence is transforming healthcare.
Tokens: ['Art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġtransforming', 'Ġhealthcare', '.']
Token count: 7

Text: Tokenization is fundamental to large language models.
Tokens: ['Token', 'ization', 'Ġis', 'Ġfundamental', 'Ġto', 'Ġlarge', 'Ġlanguage', 'Ġmodels', '.']
Token count: 9

Text: The quick brown fox jumps over the lazy dog.
Tokens: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
Token count: 10


Then ask:

Is token count equal to word count?

No.

## Part 20 — Word Count vs Token Count

### Cell 18

In [42]:
text = """
Large language models use tokenization to convert
text into a sequence of tokens that can be processed
by a neural network.
"""

word_count = len(text.split())
token_count = len(tokenizer.encode(text))

print("Word count:", word_count)
print("Token count:", token_count)

Word count: 21
Token count: 27


Then calculate:

In [43]:
ratio = token_count / word_count

print("Tokens per word:", ratio)

Tokens per word: 1.2857142857142858




This is one reason API cost and context-window capacity are expressed in tokens rather than words.

## Part 21 — Code vs Natural Language

Very useful for GenAI students.

### Cell 19

In [44]:
examples = {
    "English": "Machine learning is fascinating.",
    
    "Code": """
def calculate_average(numbers):
    return sum(numbers) / len(numbers)
""",

    "URL": "https://example.com/machine-learning/tokenization",
    
    "Emoji": "I love AI 😊🚀🔥"
}

for name, text in examples.items():

    tokens = tokenizer.tokenize(text)

    print("\n", name)
    print("Tokens:", tokens)
    print("Token count:", len(tokens))


 English
Tokens: ['Machine', 'Ġlearning', 'Ġis', 'Ġfascinating', '.']
Token count: 5

 Code
Tokens: ['Ċ', 'def', 'Ġcalculate', '_', 'average', '(', 'n', 'umbers', '):', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġreturn', 'Ġsum', '(', 'n', 'umbers', ')', 'Ġ/', 'Ġlen', '(', 'n', 'umbers', ')', 'Ċ']
Token count: 26

 URL
Tokens: ['https', '://', 'example', '.', 'com', '/', 'machine', '-', 'learning', '/', 'token', 'ization']
Token count: 12

 Emoji
Tokens: ['I', 'Ġlove', 'ĠAI', 'ĠðŁĺ', 'Ĭ', 'ðŁ', 'ļ', 'Ģ', 'ðŁ', 'Ķ', '¥']
Token count: 11


Discuss why code and URLs may tokenize into many smaller pieces.

## Part 22 — Tokenization and RAG

Now connect Lecture 5 directly to the broader GenAI course.

**Ask students:**

Suppose we have a 20-page PDF. Can we send the whole document to the LLM?

Usually not.

So we do:

```
PDF
 ↓
Text extraction
 ↓
Chunking
 ↓
Tokenization
 ↓
Embeddings
 ↓
Vector DB
```

Show the relationship with a simple experiment.

### Cell 20

In [45]:
document = """
Machine learning is a field of artificial intelligence.
Deep learning uses neural networks with multiple layers.
Transformers have become fundamental to modern NLP.
Large language models use tokenization before processing text.
Retrieval augmented generation combines retrieval with generation.
"""

Split into words:

In [47]:
words = document.split()

chunk_size = 20

chunks = [
    " ".join(words[i:i + chunk_size])
    for i in range(0, len(words), chunk_size)
]

for i, chunk in enumerate(chunks):

    token_count = len(tokenizer.encode(chunk))

    print(f"\nChunk {i + 1}")
    print(chunk)
    print("Words:", len(chunk.split()))
    print("Tokens:", token_count)


Chunk 1
Machine learning is a field of artificial intelligence. Deep learning uses neural networks with multiple layers. Transformers have become fundamental
Words: 20
Tokens: 22

Chunk 2
to modern NLP. Large language models use tokenization before processing text. Retrieval augmented generation combines retrieval with generation.
Words: 18
Tokens: 25


## Part 23 — Mini Assignment

Give students this as an in-class exercise.

### Task 1 — Character Tokenization

Implement:

In [48]:
def character_tokenize(text):
    pass

Return a list of characters.

### Task 2 — Word Tokenization

Implement:

In [49]:
def word_tokenize(text):
    pass

Handle:

`Hello, world!`

properly enough to distinguish punctuation.

### Task 3 — BPE

Modify the BPE implementation so that students can specify:

In [50]:
num_merges = 5

In [51]:
num_merges = 20

Then compare the resulting vocabulary.



What happens when the number of merges increases?

Expected discussion:
- Larger subword units
- Fewer tokens for familiar patterns
- Larger vocabulary

### Task 4 — Compare Tokenizers

Given:

In [52]:
text = """
Generative artificial intelligence is changing
the way software applications are developed.
"""

Compare:
- Character tokenizer
- Word tokenizer
- GPT-style BPE tokenizer
- BERT WordPiece tokenizer
- SentencePiece tokenizer

Create a table:

| Tokenizer | Number of tokens |
|---|---|
| Character | ? |
| Word | ? |
| BPE | ? |
| WordPiece | ? |
| SentencePiece | ? |